In [2]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score
import statsmodels.api as sm

In [3]:
# load dataset 

file_path = "/scratch/c.c2029098/dementia_ml_project/data/processed/ml_data/ml_AD_APOE_DAB1.csv"
df = pd.read_csv(file_path)

### Logistic Regression DAB1 x APOE - no interaction 

In [10]:
# Features and target
X = df[["DAB1", "APOE"]]
y = df["PHENOTYPE"]


def sim_and_split(X, y, seed=None, test_size=0.3):
    np.random.seed(seed)
    return train_test_split(X, y, test_size=test_size, stratify=y)

def run_statsmodels_logit(X_train, y_train):
    X_train = pd.DataFrame(X_train).reset_index(drop=True)
    y_train = pd.Series(np.ravel(y_train)).reset_index(drop=True)

    X_train = sm.add_constant(X_train)
    model = sm.Logit(y_train, X_train).fit(disp=False)
    return model

def run_simulations(X, y, num_reps=100):
    results = []

    for i in range(num_reps):
        seed = np.random.randint(0, 100000)
        X_train, X_test, y_train, y_test = sim_and_split(X, y, seed=seed)

        try:
            model = run_statsmodels_logit(X_train, y_train)

            X_test_df = pd.DataFrame(X_test).reset_index(drop=True)
            X_test_df = sm.add_constant(X_test_df)

            y_pred_prob = model.predict(X_test_df)
            auc = roc_auc_score(np.ravel(y_test), y_pred_prob)

            # Collect coefficients and p-values
            model_params = model.params.to_dict()
            model_pvalues = {f"{k}_pval": v for k, v in model.pvalues.items()}

            result = {"seed": seed, "AUC": auc}
            result.update(model_params)
            result.update(model_pvalues)

            results.append(result)

        except Exception as e:
            print(f"Run {i} failed (seed={seed}): {e}")
            continue

    df_results = pd.DataFrame(results)
    print(f"\nAverage AUC over {num_reps} simulations: {df_results['AUC'].mean():.4f}")
    return df_results

# Example: run the simulation
df_results = run_simulations(X, y, num_reps=100)

# Exclude non-parameter columns like 'seed' if needed
summary_stats = df_results.drop(columns=['seed']).mean()

# Print nicely formatted results
print("\nAverage values over all seeds:")
print(summary_stats.round(10))


Average AUC over 100 simulations: 0.7238

Average values over all seeds:
AUC           7.238032e-01
const         2.806632e+00
DAB1         -7.608043e-02
APOE         -1.405910e+00
const_pval    0.000000e+00
DAB1_pval     7.412253e-01
APOE_pval     3.000000e-10
dtype: float64


### Logistic Regression - DAB1 x APOE - interaction 

In [9]:
# Features and target

df["DAB1xAPOE"] = df["DAB1"] * df["APOE"]

X = df[["DAB1", "APOE", "DAB1xAPOE"]]
y = df["PHENOTYPE"]

def sim_and_split(X, y, seed=None, test_size=0.3):
    np.random.seed(seed)
    return train_test_split(X, y, test_size=test_size, stratify=y)

def run_statsmodels_logit(X_train, y_train):
    X_train = pd.DataFrame(X_train).reset_index(drop=True)
    y_train = pd.Series(np.ravel(y_train)).reset_index(drop=True)

    X_train = sm.add_constant(X_train)
    model = sm.Logit(y_train, X_train).fit(disp=False)
    return model

def run_simulations(X, y, num_reps=100):
    results = []

    for i in range(num_reps):
        seed = np.random.randint(0, 100000)
        X_train, X_test, y_train, y_test = sim_and_split(X, y, seed=seed)

        try:
            model = run_statsmodels_logit(X_train, y_train)

            X_test_df = pd.DataFrame(X_test).reset_index(drop=True)
            X_test_df = sm.add_constant(X_test_df)

            y_pred_prob = model.predict(X_test_df)
            auc = roc_auc_score(np.ravel(y_test), y_pred_prob)

            # Collect coefficients and p-values
            model_params = model.params.to_dict()
            model_pvalues = {f"{k}_pval": v for k, v in model.pvalues.items()}

            result = {"seed": seed, "AUC": auc}
            result.update(model_params)
            result.update(model_pvalues)

            results.append(result)

        except Exception as e:
            print(f"Run {i} failed (seed={seed}): {e}")
            continue

    df_results = pd.DataFrame(results)
    print(f"\nAverage AUC over {num_reps} simulations: {df_results['AUC'].mean():.4f}")
    return df_results

# Example: run the simulation
df_results = run_simulations(X, y, num_reps=100)

# Exclude non-parameter columns like 'seed' if needed
summary_stats = df_results.drop(columns=['seed']).mean()

# Print nicely formatted results
print("\nAverage values over all seeds:")
print(summary_stats.round(10))


Average AUC over 100 simulations: 0.7194

Average values over all seeds:
AUC               7.194354e-01
const             2.709122e+00
DAB1              1.051795e+00
APOE             -1.343719e+00
DAB1xAPOE        -6.834468e-01
const_pval        0.000000e+00
DAB1_pval         5.414683e-01
APOE_pval         1.140000e-08
DAB1xAPOE_pval    4.816069e-01
dtype: float64


### Logistic Regression - APOE only 

In [7]:
# Features and target
X = df[["APOE"]]
y = df["PHENOTYPE"]


def sim_and_split(X, y, seed=None, test_size=0.3):
    np.random.seed(seed)
    return train_test_split(X, y, test_size=test_size, stratify=y)

def run_statsmodels_logit(X_train, y_train):
    X_train = pd.DataFrame(X_train).reset_index(drop=True)
    y_train = pd.Series(np.ravel(y_train)).reset_index(drop=True)

    X_train = sm.add_constant(X_train)
    model = sm.Logit(y_train, X_train).fit(disp=False)
    return model

def run_simulations(X, y, num_reps=100):
    results = []

    for i in range(num_reps):
        seed = np.random.randint(0, 100000)
        X_train, X_test, y_train, y_test = sim_and_split(X, y, seed=seed)

        try:
            model = run_statsmodels_logit(X_train, y_train)

            X_test_df = pd.DataFrame(X_test).reset_index(drop=True)
            X_test_df = sm.add_constant(X_test_df)

            y_pred_prob = model.predict(X_test_df)
            auc = roc_auc_score(np.ravel(y_test), y_pred_prob)

            # Collect coefficients and p-values
            model_params = model.params.to_dict()
            model_pvalues = {f"{k}_pval": v for k, v in model.pvalues.items()}

            result = {"seed": seed, "AUC": auc}
            result.update(model_params)
            result.update(model_pvalues)

            results.append(result)

        except Exception as e:
            print(f"Run {i} failed (seed={seed}): {e}")
            continue

    df_results = pd.DataFrame(results)
    print(f"\nAverage AUC over {num_reps} simulations: {df_results['AUC'].mean():.4f}")
    return df_results

# Example: run the simulation
df_results = run_simulations(X, y, num_reps=100)

# Exclude non-parameter columns like 'seed' if needed
summary_stats = df_results.drop(columns=['seed']).mean()

# Print nicely formatted results
print("\nAverage values over all seeds:")
print(summary_stats.round(10))


Average AUC over 100 simulations: 0.7087

Average values over all seeds:
AUC           7.087393e-01
const         2.884658e+00
APOE         -1.472283e+00
const_pval    0.000000e+00
APOE_pval     1.000000e-09
dtype: float64


### Logistic Regression - DAB1 only 

In [8]:
X = df[["DAB1"]]
y = df["PHENOTYPE"]

def sim_and_split(X, y, seed=None, test_size=0.3):
    np.random.seed(seed)
    return train_test_split(X, y, test_size=test_size, stratify=y)

def run_statsmodels_logit(X_train, y_train):
    X_train = pd.DataFrame(X_train).reset_index(drop=True)
    y_train = pd.Series(np.ravel(y_train)).reset_index(drop=True)

    X_train = sm.add_constant(X_train)
    model = sm.Logit(y_train, X_train).fit(disp=False)
    return model

def run_simulations(X, y, num_reps=100):
    results = []

    for i in range(num_reps):
        seed = np.random.randint(0, 100000)
        X_train, X_test, y_train, y_test = sim_and_split(X, y, seed=seed)

        try:
            model = run_statsmodels_logit(X_train, y_train)

            X_test_df = pd.DataFrame(X_test).reset_index(drop=True)
            X_test_df = sm.add_constant(X_test_df)

            y_pred_prob = model.predict(X_test_df)
            auc = roc_auc_score(np.ravel(y_test), y_pred_prob)

            # Collect coefficients and p-values
            model_params = model.params.to_dict()
            model_pvalues = {f"{k}_pval": v for k, v in model.pvalues.items()}

            result = {"seed": seed, "AUC": auc}
            result.update(model_params)
            result.update(model_pvalues)

            results.append(result)

        except Exception as e:
            print(f"Run {i} failed (seed={seed}): {e}")
            continue

    df_results = pd.DataFrame(results)
    print(f"\nAverage AUC over {num_reps} simulations: {df_results['AUC'].mean():.4f}")
    return df_results

# Example: run the simulation
df_results = run_simulations(X, y, num_reps=100)

# Exclude non-parameter columns like 'seed' if needed
summary_stats = df_results.drop(columns=['seed']).mean()

# Print nicely formatted results
print("\nAverage values over all seeds:")
print(summary_stats.round(10))




Average AUC over 100 simulations: 0.4859

Average values over all seeds:
AUC           4.858859e-01
const         7.785462e-01
DAB1         -4.862987e-02
const_pval    1.000000e-10
DAB1_pval     7.166878e-01
dtype: float64
